In [ ]:
import os
import pandas as pd
import requests
import time
from tqdm import tqdm

API_KEY = " "
API_URL = "https://api.deepseek.com/v1/chat/completions"

questions = [
    {"type": "TREATMENT", "question": "What treats Chronic Obstructive Pulmonary Disease?"},
    {"type": "TREATMENT", "question": "What can slow the progression of Chronic Obstructive Pulmonary Disease?"},
    {"type": "TREATMENT", "question": "What can lower the risk of Chronic Obstructive Pulmonary Disease?"},
    {"type": "TREATMENT", "question": "What prevents Chronic Obstructive Pulmonary Disease?"},
    {"type": "TREATMENT", "question": "What inhibits Chronic Obstructive Pulmonary Disease?"},
    {"type": "FACTOR", "question": "What causes Chronic Obstructive Pulmonary Disease?"},
    {"type": "FACTOR", "question": "What does cause Chronic Obstructive Pulmonary Disease?"},
    {"type": "FACTOR", "question": "What are the risk factors for Chronic Obstructive Pulmonary Disease?"},
    {"type": "FACTOR", "question": "What can increase the risk of Chronic Obstructive Pulmonary Disease?"},
    {"type": "FACTOR", "question": "What can result in Chronic Obstructive Pulmonary Disease?"},
    {"type": "COEXISTS", "question": "What coexists with Chronic Obstructive Pulmonary Disease?"},
    {"type": "COEXISTS", "question": "What can Chronic Obstructive Pulmonary Disease convert to?"},
    {"type": "COEXISTS", "question": "What does Chronic Obstructive Pulmonary Disease lead to?"},
    {"type": "COEXISTS", "question": "What can be caused by Chronic Obstructive Pulmonary Disease?"},
    {"type": "COEXISTS", "question": "What can be associated with Chronic Obstructive Pulmonary Disease?"}
]

def call_deepseek_api(prompt, max_retries=3):
    headers = {
        "Authorization": f"Bearer {API_KEY}",
        "Content-Type": "application/json"
    }
    data = {
        "model": "deepseek-chat",
        "messages": [{"role": "user", "content": prompt}],
        "max_tokens": 200,
        "temperature": 0.1,
    }
    
    for attempt in range(max_retries):
        try:
            response = requests.post(API_URL, headers=headers, json=data, timeout=30)
            if response.status_code == 200:
                result = response.json()
                return result['choices'][0]['message']['content'].strip()
            elif response.status_code == 429:
                wait_time = 2 ** attempt
                print(f"⚠️ Rate limit hit, waiting {wait_time} seconds...")
                time.sleep(wait_time)
            else:
                return f"API Error: HTTP {response.status_code}"
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            else:
                return f"Request Exception: {str(e)}"
    return "Max retries exceeded"

def save_to_excel(all_results, filename="copd_answers_chunked.xlsx"):
    if not all_results:
        return
    treatment_data = []
    factor_data = []
    coexists_data = []
    for record in all_results:
        row_id = record["Row_ID"]
        note_sample = record["Clinical_Note_Sample"]
        for key, value in record.items():
            if key.startswith("Q") and not key.endswith(("ID", "Sample")):
                if "(TREATMENT)" in key:
                    treatment_data.append({
                        "Row_ID": row_id,
                        "Clinical_Note_Sample": note_sample,
                        "Question": key,
                        "LLM_Answer": value
                    })
                elif "(FACTOR)" in key:
                    factor_data.append({
                        "Row_ID": row_id,
                        "Clinical_Note_Sample": note_sample,
                        "Question": key,
                        "LLM_Answer": value
                    })
                elif "(COEXISTS)" in key:
                    coexists_data.append({
                        "Row_ID": row_id,
                        "Clinical_Note_Sample": note_sample,
                        "Question": key,
                        "LLM_Answer": value
                    })
    df_treatment = pd.DataFrame(treatment_data) if treatment_data else pd.DataFrame(columns=["Row_ID", "Clinical_Note_Sample", "Question", "LLM_Answer"])
    df_factor = pd.DataFrame(factor_data) if factor_data else pd.DataFrame(columns=["Row_ID", "Clinical_Note_Sample", "Question", "LLM_Answer"])
    df_coexists = pd.DataFrame(coexists_data) if coexists_data else pd.DataFrame(columns=["Row_ID", "Clinical_Note_Sample", "Question", "LLM_Answer"])
    try:
        with pd.ExcelWriter(filename, engine='openpyxl') as writer:
            df_treatment.to_excel(writer, sheet_name='TREATMENT', index=False)
            df_factor.to_excel(writer, sheet_name='FACTOR', index=False)
            df_coexists.to_excel(writer, sheet_name='COEXISTS', index=False)
        print(f"💾 Checkpoint Saved: {filename} (TREATMENT, FACTOR, COEXISTS sheets updated)")
    except Exception as e:
        print(f"❌ Error saving file {filename}: {e}")

def process_clinical_notes(excel_file_path, output_file="C:/Users/ASUS/Desktop/copd_answers_chunked.xlsx", save_interval=10):
    print(f"📁 Starting processing. Input: {excel_file_path}")
    
    try:
        df_input = pd.read_excel(excel_file_path,sheet_name=1)
        
        if df_input.shape[1] < 4:
            raise ValueError("Input file does not have at least 4 columns.")
        try:
            clinical_notes_series = df_input['full_text']
        except KeyError:
            clinical_notes_series = df_input.iloc[:, 3]
        print(f"✅ Loaded {len(clinical_notes_series)} records from column 4")
    except Exception as e:
        print(f"❌ Error loading Excel file: {e}")
        return

    all_results = []

    print(f"\n🚀 Starting processing... Will save every {save_interval} records.")

    for idx, row in tqdm(enumerate(clinical_notes_series.items()), total=len(clinical_notes_series), desc="Processing"):
        row_id = row[0] 
        clinical_note = str(row[1])[:3000] 
    
        result_row = {
            "Row_ID": row_id, 
            "Clinical_Note_Sample": clinical_note[:100]
        }

        for i, q in enumerate(questions, 1):
            q_type = q["type"]
            q_text = q["question"]
            
            prompt = f"""You are a medical knowledge extraction assistant. Your task is to extract specific entities from the clinical note.

### Input
question_type: {q_type}
question: {q_text}
context: {clinical_note}

### Instructions
1. **Strict Output Format**: You must output exactly in this format: {q_type}: Entity1, Entity2, Entity3
2. **Entities Only**: Extract only the keywords/phrases mentioned in the context.
3. **No Explanations**: Do not write sentences, do not write "I don't know", do not write "Based on the note".
4. **No Hallucination**: If the entity is not found, output: {q_type}: Not found

### Response """
            
            raw_answer = call_deepseek_api(prompt)
            
            result_row[f"Q{i} ({q_type})"] = raw_answer
            
            time.sleep(0.5) 

        all_results.append(result_row)

        if (idx + 1) % save_interval == 0:
            save_to_excel(all_results, output_file)
            print(f"📦 Processed {idx + 1} records...")

    if all_results:
        save_to_excel(all_results, output_file)
        print(f"\n🎉 Final Processing completed! Total records processed: {len(all_results)}")
        print(f"📄 Final results saved to {output_file}")

if __name__ == "__main__":
    input_excel_file = "./copd_matched_records1.xlsx" 
    if not os.path.exists(input_excel_file):
        print(f"❌ Input file '{input_excel_file}' not found. Please place your file in the same directory.")
    else:
        process_clinical_notes(input_excel_file, "./copd_answers_chunked.xlsx", 10)